# Receive OpenLineage events from Airflow

Runs the *HarvestOpenLineageEvents::CreateAsCatalogTargetGovernanceActionProcess* governance action process. It catalogs the `openlineage.events` Kafka topic and makes it a catalog target of the **OpenLineageKafkaListener** integration connector, which then passes each event to the **OpenLineageCataloguer** to record as lineage.

Airflow sends its events to the OpenLineage proxy backend in shared-infra (port 6000), which publishes them to that topic. The host and port are those Egeria uses to reach Kafka: the shared Kafka server's external listener, `host.docker.internal:9194` (override with `OPENLINEAGE_KAFKA_HOST` / `OPENLINEAGE_KAFKA_PORT`).

The Portal's bootstrap monitor runs this notebook after a demo reset (see `_batch.json`). The step-by-step version is `workbooks/cataloguing-and-surveys/apache-kafka/kafka-open-lineage-events.ipynb`.

**Run it once.** Each run of the process adds another catalog target, and the listener then reads every event once per target (on 2026-11-12 a re-heal loop left 21 on one topic). So the notebook first checks whether a `Topic` named `openlineage.events` already exists **and** is already a catalog target of the OpenLineageKafkaListener, and creates nothing if so. If it cannot tell (an error, or any answer other than Egeria's own *No elements found*), it refuses, in words, and creates nothing.

In [ ]:
import os

from pyegeria import AutomatedCuration, MetadataExpert

view_server = os.environ.get("EGERIA_VIEW_SERVER", "qs-view-server")
url = os.environ.get("EGERIA_PLATFORM_URL", "https://localhost:9443")
user_id = os.environ.get("EGERIA_USER", "erinoverview")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD", "secret")

automated_curation = AutomatedCuration(view_server, url, user_id, user_pwd)
token = automated_curation.create_egeria_bearer_token(user_id, user_pwd)
# The metadata expert runs the existence check (the same find as the Portal's bootstrap monitor).
metadata_expert = MetadataExpert(view_server, url, user_id, user_pwd)
metadata_expert.set_bearer_token(token)


In [ ]:
# Is openlineage.events already catalogued AND already a catalog target of the listener?

TOPIC_DISPLAY_NAME = "openlineage.events"
LISTENER_GUID = "980b989c-de78-4e6a-a58d-51049d7381bf"     # OpenLineageKafkaListener (integration connector)
LISTENER_NAME = "OpenLineageKafkaListener"
NO_ELEMENTS_FOUND = "No elements found"                       # pyegeria's answer when Egeria found none
NONE_ANSWERS = (NO_ELEMENTS_FOUND, "No elements returned")
# A page of 25, never 1: Egeria 8840035f answers a find with pageSize 1 with "No elements found" even
# when a match exists (bug B5, 2026-11-12), which read a catalogued topic as absent.
PAGE_SIZE = 25

def find_topics():
    """Every active Topic whose display name is exactly openlineage.events (the Portal monitor's find)."""
    return metadata_expert.find_metadata_elements({
        "class": "FindRequestBody", "metadataElementTypeName": "Topic",
        "searchProperties": {"class": "SearchProperties", "matchCriteria": "ALL",
                             "conditions": [{"property": "displayName", "operator": "EQ",
                                             "value": {"class": "PrimitiveTypePropertyValue", "typeName": "string",
                                                       "primitiveValue": TOPIC_DISPLAY_NAME}}]},
        "limitResultsByStatus": ["ACTIVE"], "graphQueryDepth": 0, "startFrom": 0, "pageSize": PAGE_SIZE})

def catalog_targets_of(topic_guid):
    """The topic's CatalogTarget relationships (the topic is end 2; the integration connector end 1)."""
    return metadata_expert.get_related_metadata_elements(
        topic_guid, "CatalogTarget", {"class": "ResultsRequestBody", "limitResultsByStatus": ["ACTIVE"]},
        starting_at_end=2, start_from=0, page_size=PAGE_SIZE)

def _strings(obj, out):
    """Every string value in a pyegeria answer (GUIDs, qualified and display names)."""
    if isinstance(obj, dict):
        for v in obj.values():
            _strings(v, out)
    elif isinstance(obj, list):
        for v in obj:
            _strings(v, out)
    elif isinstance(obj, str):
        out.append(obj)
    return out

def _listener_links(related):
    """How many of the related elements are the listener (one per CatalogTarget relationship)."""
    items = related.get("elementList") if isinstance(related, dict) else related
    if not isinstance(items, list):
        items = [related]
    return sum(1 for item in items
               if any(s == LISTENER_GUID or LISTENER_NAME in s for s in _strings(item, [])))

def check_listener():
    """("connected", [(qualifiedName, GUID, catalog targets)]), ("none", reason) -- create it -- or
    ("not read", reason) -- create nothing. Only Egeria's own "No elements found" means there is none."""
    try:
        found = find_topics()
    except Exception as error:
        return "not read", f"the topic lookup failed: {type(error).__name__}: {error}"
    if found == NO_ELEMENTS_FOUND:
        return "none", f"no Topic named {TOPIC_DISPLAY_NAME}"
    if not isinstance(found, list) or not found:
        return "not read", f"the topic lookup gave an unexpected answer: {str(found)[:120]!r}"
    connected, unconnected = [], []
    for element in found:
        if not isinstance(element, dict) or not element.get("elementGUID"):
            return "not read", f"a topic in the answer has no GUID: {str(element)[:120]!r}"
        guid = element["elementGUID"]
        props = (element.get("elementProperties") or {}).get("propertyValueMap") or {}
        qualified_name = (props.get("qualifiedName") or {}).get("primitiveValue") or "?"
        try:
            related = catalog_targets_of(guid)
        except Exception as error:
            return "not read", f"the catalog targets of {qualified_name} could not be read: {type(error).__name__}: {error}"
        if related in NONE_ANSWERS:
            unconnected.append(qualified_name)
            continue
        if not isinstance(related, (dict, list)):
            return "not read", f"the catalog targets of {qualified_name} gave an unexpected answer: {str(related)[:120]!r}"
        links = _listener_links(related)
        (connected if links else unconnected).append((qualified_name, guid, links) if links else qualified_name)
    if connected:
        return "connected", connected
    return "none", f"{TOPIC_DISPLAY_NAME} exists ({', '.join(unconnected)}) but is not a catalog target of {LISTENER_NAME}"

listener_state, listener_detail = check_listener()
if listener_state == "connected":
    for qualified_name, guid, links in listener_detail:
        print(f"{qualified_name} ({guid}) is already a catalog target of {LISTENER_NAME}"
              + (f" {links} times: duplicates to clean up, each event is read {links} times" if links > 1 else "")
              + "; nothing is created.")
elif listener_state == "not read":
    raise RuntimeError(f"Could not read whether {TOPIC_DISPLAY_NAME} is already a catalog target of "
                       f"{LISTENER_NAME} ({listener_detail}); not creating one.")
else:
    print(f"{listener_detail}: starting the process.")


In [ ]:
if listener_state == "none":
    request_parameters = {
        "serverName": "LocalKafka2",
        "hostIdentifier": os.environ.get("OPENLINEAGE_KAFKA_HOST", "host.docker.internal"),
        "portNumber": os.environ.get("OPENLINEAGE_KAFKA_PORT", "9194"),
        "description": "Receive open lineage events via Kafka Topic.",
        "topicDescription": "Open Lineage Events for Backend Proxy",
        "versionIdentifier": "V1.0",
        "fullTopicName": "openlineage.events",
        "shortTopicName": "openlineage.events",
        "eventDirection": "inOut",
    }

    process_guid = automated_curation.initiate_gov_action_process(
        "HarvestOpenLineageEvents::CreateAsCatalogTargetGovernanceActionProcess",
        None, None, None, request_parameters, None, None,
    )
    print(f"Started HarvestOpenLineageEvents: {process_guid}")
